## how to use this notebook

this notebook implements **System 2** in the pooled evaluation pipeline for filtering
social-policy legislation from the German BGBl corpus. System 1 (cosine similarity) is
in `step3_process_germany_main.ipynb`; the union of both systems is compiled in
`step_pooled_union_compile.py`.

**workflow:**

1. set `INPUT_FILES` and `OUTPUT_PREFIX` in the **configuration** cell to match your corpus
   and the prefix used in step 3 / step 4
2. run §1–§2 to load data and embeddings (cached from step 3 if available, otherwise recomputed)
3. run §3 to fit BERTopic and inspect the topic count
4. run §4 to print the topic table (topic_id, document count, top words) and save an interactive
   HTML visualisation — **skim the top-words column and decide which topic_ids correspond to
   social policy**
5. fill in `SOCIAL_POLICY_TOPICS` in §5 with those topic_id integers
6. run §6–§7 to extract candidates, handle the noise pool, and save all outputs

**outputs saved to `OUTPUT_DIR`:**
- `{OUTPUT_PREFIX}_berttopic_candidates.json` — documents from social-policy topics → System 2 candidates
- `{OUTPUT_PREFIX}_berttopic_noise_pool.json` — documents assigned to BERTopic noise topic (-1) → spot-check sample
- `runlogs_from_berttopic/berttopic_runlog_{OUTPUT_PREFIX}_{date}.md` — run log
- `runlogs_from_berttopic/berttopic_topics_{OUTPUT_PREFIX}_{date}.html` — interactive topic visualisation

# step 2.2 — system 2: BERTopic clustering

In [ ]:
# pip installs (run once):
# pip install bertopic umap-learn hdbscan sentence-transformers

In [1]:
import gzip
import json
import datetime
import os
import stat
import numpy as np
import nltk
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords
from pathlib import Path
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
import pandas as pd

In [10]:
# ── CONFIGURATION — set these before running ──────────────────────────────────

def _find_root(marker="CLAUDE.md"):
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / marker).exists():
            return p
    raise FileNotFoundError(f"project root not found (no {marker} above {Path.cwd()})")

PROJECT_ROOT = _find_root()

INPUT_FILE    = PROJECT_ROOT / "data/raw/germany/bgbl1_2008-2015_2019-2022_combined.json.gz"
OUTPUT_DIR    = PROJECT_ROOT / "data/processed/germany"
OUTPUT_PREFIX = "germany_2008-2015_2019-2022"   # must match step 3 / step 4 prefix

MODEL_NAME      = "paraphrase-multilingual-mpnet-base-v2"  # same model as System 1
RANDOM_SEED     = 42
STOPWORDS_LANG  = "german"   # choose any language supported by nltk.corpus.stopwords

# embedding cache — set to None to always recompute
EMBEDDING_CACHE     = OUTPUT_DIR / f".embed_cache_{OUTPUT_PREFIX}.npy"
EMBEDDING_CACHE_IDX = OUTPUT_DIR / f".embed_cache_{OUTPUT_PREFIX}_titles.json"

# BERTopic parameters — tune after inspecting first run
MIN_CLUSTER_SIZE = 20   # HDBSCAN: minimum docs per cluster
N_NEIGHBORS      = 50   # UMAP: neighbourhood size
N_COMPONENTS     = 5    # UMAP: dimensionality for BERTopic (not for visualisation)

# domain-specific stopwords added on top of the NLTK stopword list
EXTRA_STOPWORDS = [
    "gesetz", "verordnung", "aenderung", "artikel",
    "bundesgesetzblatt", "bekanntmachung",
]

# ─────────────────────────────────────────────────────────────────────────────

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")
print(f"Output dir:   {OUTPUT_DIR}")

Project root: s:\stbuchho\crises_and_welfare_states\policy-tracker
Output dir:   s:\stbuchho\crises_and_welfare_states\policy-tracker\data\processed\germany


## 1. load data

In [3]:
with gzip.open(INPUT_FILE, "rt", encoding="utf-8") as fh:
    raw = json.load(fh)

data = raw if isinstance(raw, list) else list(raw.values())
print(f"{len(data):>5} docs  ←  {INPUT_FILE.name}")

titles = [doc["title"] for doc in data]
print(f"\nTotal corpus: {len(data)} documents")
print("\nSample titles:")
for t in titles[:5]:
    print(f"  {t[:120]}")

 5048 docs  ←  bgbl1_2008-2015_2019-2022_combined.json.gz

Total corpus: 5048 documents

Sample titles:
  Gesetz zur Einführung einer Pflicht zur Mitteilung grenzüberschreitender Steuergestaltungen
  Gesetz zur Umsetzung des Klimaschutzprogramms 2030 im Steuerrecht
  Gesetz über die Feststellung des Bundeshaushaltsplans für das Haushaltsjahr 2020 (Haushaltsgesetz 2020)
  Gesetz zur Verlängerung des Betrachtungszeitraums für die ortsübliche Vergleichsmiete
  Gesetz zur Einführung eines Freibetrages in der gesetzlichen Krankenversicherung zur Förderung der betrieblichen Altersv


## 2. load or compute embeddings

uses the same model as System 1 (`step3_process_germany_main.ipynb`). if the embedding
cache exists and matches the current title list, it is loaded directly. otherwise embeddings
are computed and saved for future reuse.

In [4]:
def _compute_and_cache(titles):
    model = SentenceTransformer(MODEL_NAME)
    emb = model.encode(titles, batch_size=16, show_progress_bar=True)
    np.save(EMBEDDING_CACHE, emb)
    EMBEDDING_CACHE_IDX.write_text(
        json.dumps(titles, ensure_ascii=False), encoding="utf-8"
    )
    return emb

if EMBEDDING_CACHE and EMBEDDING_CACHE.exists() and EMBEDDING_CACHE_IDX.exists():
    cached_titles = json.loads(EMBEDDING_CACHE_IDX.read_text(encoding="utf-8"))
    if cached_titles == titles:
        doc_embeddings = np.load(EMBEDDING_CACHE)
        print(f"Loaded cached embeddings: {doc_embeddings.shape}")
    else:
        print("Cache title mismatch — recomputing")
        doc_embeddings = _compute_and_cache(titles)
        print(f"Computed and cached: {doc_embeddings.shape}")
else:
    doc_embeddings = _compute_and_cache(titles)
    print(f"Computed and cached: {doc_embeddings.shape}")

Loaded cached embeddings: (5048, 768)


## 3. fit BERTopic

UMAP reduces the embedding dimensions before HDBSCAN clusters them. the vectorizer
then computes c-TF-IDF top words per cluster for interpretation.

adjust `MIN_CLUSTER_SIZE` (configuration cell) if you get too many / too few topics:
smaller → more, finer-grained topics; larger → fewer, broader topics.

In [11]:
umap_model = UMAP(
    n_neighbors=N_NEIGHBORS,
    n_components=N_COMPONENTS,
    min_dist=0.0,
    metric="cosine",
    random_state=RANDOM_SEED,
)
hdbscan_model = HDBSCAN(
    min_cluster_size=MIN_CLUSTER_SIZE,
    min_samples=5,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True,
)
_stop_words = stopwords.words(STOPWORDS_LANG) + EXTRA_STOPWORDS
vectorizer = CountVectorizer(ngram_range=(1, 2), min_df=2, stop_words=_stop_words)

topic_model = BERTopic(
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer,
    calculate_probabilities=False,
    verbose=True,
)

topics, _ = topic_model.fit_transform(titles, embeddings=doc_embeddings)

topic_info = topic_model.get_topic_info()
n_topics   = topic_info.shape[0] - 1   # exclude noise topic
n_noise    = int((np.array(topics) == -1).sum())

print(f"\nTopics found:          {n_topics}")
print(f"Noise docs (topic -1): {n_noise}")

2026-05-21 18:21:13,037 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-21 18:21:23,900 - BERTopic - Dimensionality - Completed ✓
2026-05-21 18:21:23,902 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-21 18:21:24,014 - BERTopic - Cluster - Completed ✓
2026-05-21 18:21:24,017 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-21 18:21:24,131 - BERTopic - Representation - Completed ✓



Topics found:          76
Noise docs (topic -1): 1077


## 4. inspect topics

skim the **Representation** column for each topic. identify which `Topic` ids correspond
to social-policy legislation, then enter those ids in the configuration cell in §5.

topic `-1` is the noise bucket — documents HDBSCAN could not assign to any cluster.
they are saved separately as a spot-check pool.

In [12]:
#requires import pandas
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 200)

display(topic_info[["Topic", "Count", "Name"]].sort_values("Topic"))

,Topic,Count,Name
0,-1,1077,-1_fr_rentenversicherung_durchfhrung_fr jahr
1,0,369,0_berufsausbildung_ber berufsausbildung_anerkannten_prfung anerkannten
2,1,230,1_verordnungen_zweite nderung_zweite_nderung
3,2,122,2_straenverkehr_vorschriften_ausnahmen_ber ausnahmen
4,3,122,3_seeschifffahrt_see_gefahrgutverordnung_binnenschifffahrt
5,4,112,4_futtermittelverordnung_nderung futtermittelverordnung_lebensmittel_lebensmittel futtermittelgesetzbuches
6,5,89,5_drittes nderung_drittes_notariellen_abgeordnetengesetzes
7,6,88,6_energiewirtschaftsgesetzes_strom_energie_neuregelung
8,7,87,7_bundestages_deutschen bundestages_geschftsordnung_nderung geschftsordnung
9,8,87,8_arzneimittel_arzneimittelgesetz_arzneimitteln_arzneimittelrechtlicher


In [ ]:
# interactive bar chart — top words per topic (renders inline)
topic_model.visualize_barchart(top_n_topics=n_topics, n_words=8, height=250)

In [ ]:
# save interactive topic scatter plot as HTML for external inspection
log_dir = OUTPUT_DIR / "runlogs_from_berttopic"
log_dir.mkdir(parents=True, exist_ok=True)

_run_date = datetime.date.today().isoformat()
_viz_path = log_dir / f"berttopic_topics_{OUTPUT_PREFIX}_{_run_date}.html"

fig_topics = topic_model.visualize_topics()
fig_topics.write_html(_viz_path)
print(f"Interactive topic map saved: {_viz_path}")

# also print full topic table to stdout for quick skimming
print()
for _, row in topic_info.sort_values("Topic").iterrows():
    tid = int(row["Topic"])
    label = "[NOISE]" if tid == -1 else f"topic {tid:>3}"
    words = row["Representation"] if isinstance(row.get("Representation"), list) else []
    print(f"  {label}  n={int(row['Count']):>4}   {', '.join(words[:8])}")

## 5. label social-policy topics  ← fill in after inspecting §4

enter the `topic_id` integers from the table above that correspond to social-policy legislation.
only run §6 and §7 after filling this in.

In [ ]:
# ── FILL IN after inspecting §4 ───────────────────────────────────────────────

SOCIAL_POLICY_TOPICS = []   # example: [0, 3, 7, 12]

# ─────────────────────────────────────────────────────────────────────────────

if SOCIAL_POLICY_TOPICS:
    print(f"Labelled {len(SOCIAL_POLICY_TOPICS)} topic(s) as social policy: {SOCIAL_POLICY_TOPICS}")
    for tid in SOCIAL_POLICY_TOPICS:
        words = [w for w, _ in topic_model.get_topic(tid)[:8]]
        count = int(topic_info.loc[topic_info["Topic"] == tid, "Count"].values[0])
        print(f"  topic {tid:>3}  n={count:>4}   {', '.join(words)}")
else:
    print("SOCIAL_POLICY_TOPICS is empty — fill in above before running §6 and §7.")

## 6. extract candidates and noise pool

In [ ]:
assert SOCIAL_POLICY_TOPICS, (
    "SOCIAL_POLICY_TOPICS is empty — fill in §5 before running this cell."
)

candidates = []
noise_pool = []

for i, doc in enumerate(data):
    t = topics[i]
    top_words = (
        [w for w, _ in topic_model.get_topic(t)[:5]] if t != -1 else []
    )
    entry = {**doc, "berttopic_topic": t, "berttopic_top_words": top_words}
    if t in SOCIAL_POLICY_TOPICS:
        candidates.append(entry)
    elif t == -1:
        noise_pool.append(entry)

other = len(data) - len(candidates) - len(noise_pool)
print(f"Candidates (selected topics): {len(candidates)}")
print(f"Noise pool (topic -1):        {len(noise_pool)}")
print(f"Other topics (not selected):  {other}")
print(f"Total:                        {len(data)}")

## 7. save

In [ ]:
cand_path  = OUTPUT_DIR / f"{OUTPUT_PREFIX}_berttopic_candidates.json"
noise_path = OUTPUT_DIR / f"{OUTPUT_PREFIX}_berttopic_noise_pool.json"

cand_path.write_text(
    json.dumps(candidates, ensure_ascii=False, indent=2), encoding="utf-8"
)
noise_path.write_text(
    json.dumps(noise_pool, ensure_ascii=False, indent=2), encoding="utf-8"
)

print(f"Saved: {cand_path}")
print(f"Saved: {noise_path}")
print("\nNext: run step_pooled_union_compile.py to merge System 1 and System 2 candidates.")

In [ ]:
run_date = datetime.date.today().isoformat()
log_path = log_dir / f"berttopic_runlog_{OUTPUT_PREFIX}_{run_date}.md"

if log_path.exists():
    os.chmod(log_path, stat.S_IWRITE)

selected_topic_rows = []
for tid in sorted(SOCIAL_POLICY_TOPICS):
    words = [w for w, _ in topic_model.get_topic(tid)[:8]]
    count = int(topic_info.loc[topic_info["Topic"] == tid, "Count"].values[0])
    selected_topic_rows.append(f"| {tid} | {count} | {', '.join(words)} |")

content = f"""# BERTopic run log

| | |
|---|---|
| date | {run_date} |
| input file | {INPUT_FILE.name} |
| output prefix | `{OUTPUT_PREFIX}` |
| model | `{MODEL_NAME}` |

## BERTopic parameters

| parameter | value |
|---|---|
| MIN_CLUSTER_SIZE | {MIN_CLUSTER_SIZE} |
| N_NEIGHBORS | {N_NEIGHBORS} |
| N_COMPONENTS | {N_COMPONENTS} |

## results

| metric | n |
|---|---|
| total documents | {len(data)} |
| topics found | {n_topics} |
| noise docs (topic -1) | {n_noise} |
| social-policy topics labelled | {len(SOCIAL_POLICY_TOPICS)} |
| candidates (System 2) | {len(candidates)} |
| noise pool | {len(noise_pool)} |

## labelled social-policy topics

| topic_id | count | top words |
|---|---|---|
{chr(10).join(selected_topic_rows)}

## output files

- `{cand_path}`
- `{noise_path}`
- `{_viz_path}`
"""

log_path.write_text(content, encoding="utf-8")
os.chmod(log_path, stat.S_IREAD | stat.S_IRGRP | stat.S_IROTH)
print(f"Run log saved (read-only): {log_path}")